# chrome-ocr demo

All test images and PDFs are generated in-memory — no external files needed.

**Requirements**
```
pip install pillow numpy protobuf          # image OCR
pip install pymupdf                        # PDF support (optional)
```
Google Chrome must be installed (Screen AI component is downloaded automatically
by Chrome for its accessibility features).

In [ ]:
import io, sys, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

sys.path.insert(0, '..')   # only needed when running from the example/ subdirectory

from chrome_ocr import ocr_pdf, ocr_img, ScreenAIEngine

# -------------------------------------------------------------------
# Create ONE engine and reuse it everywhere.
# chrome_screen_ai.dll must be initialised only once per process;
# creating a second ScreenAIEngine() previously caused a kernel crash.
# chrome_ocr now guards against this automatically, but it's still best
# practice to instantiate once and pass engine= to every call.
# -------------------------------------------------------------------
ENGINE = ScreenAIEngine()
print(f'Engine ready : {ENGINE.ok}')
print(f'Max image dim: {ENGINE.max_dimension} px')

## Helper — font loader

Tries system fonts (Arial on Windows, FreeSans/Liberation on Linux) before
falling back to PIL's built-in bitmap font.
A real TrueType font gives the OCR engine clean, well-spaced glyphs.

In [ ]:
from PIL import ImageFont

def load_font(size: int):
    for path in [
        'arial.ttf',
        'Arial.ttf',
        '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    ]:
        try:
            return ImageFont.truetype(path, size)
        except (OSError, IOError):
            pass
    return ImageFont.load_default()

print('Font loader ready')

---
## 1 — ocr_img(): file path input

We render a realistic multi-section document image and run OCR on it.
The image is saved to disk to demonstrate the file-path code path.

In [ ]:
from PIL import Image, ImageDraw

W, H = 900, 500
img = Image.new('RGB', (W, H), 'white')
d   = ImageDraw.Draw(img)

d.text((40,  25), 'Sample Document — chrome-ocr',                      fill='black', font=load_font(36))
d.text((40,  90), 'Section 1: Introduction',                            fill='black', font=load_font(22))
d.text((40, 125), 'chrome-ocr converts images and PDFs to Markdown.',   fill='#111',  font=load_font(18))
d.text((40, 155), 'It calls Chrome built-in Screen AI engine via ctypes.', fill='#111', font=load_font(18))
d.text((40, 205), 'Section 2: Features',                                fill='black', font=load_font(22))
d.text((40, 240), '- Layout-aware output: headings, tables, math',      fill='#111',  font=load_font(18))
d.text((40, 270), '- Accepts file path, PIL Image, or NumPy array',     fill='#111',  font=load_font(18))
d.text((40, 300), '- No API key, no network, no separate model',        fill='#111',  font=load_font(18))
d.text((40, 350), 'Section 3: Quick start',                             fill='black', font=load_font(22))
d.text((40, 385), 'from chrome_ocr import ocr_pdf, ocr_img', fill='#333', font=load_font(16))
d.text((40, 415), 'text = ocr_img("scan.png")',                   fill='#333',  font=load_font(16))
d.text((40, 445), 'md   = ocr_pdf("report.pdf")',               fill='#333',  font=load_font(16))

img.save('test_image.png')
print(f'Image saved: {W}x{H} px')

In [ ]:
# OCR via file path — note engine= is passed explicitly
result = ocr_img('test_image.png', engine=ENGINE)
print(result)

---
## 2 — ocr_img(): PIL Image and NumPy array inputs

In [ ]:
import numpy as np

print('--- PIL Image ---')
print(ocr_img(img, engine=ENGINE))

print('\n--- NumPy array (RGB uint8) ---')
print(ocr_img(np.array(img), engine=ENGINE))

---
## 3 — ocr_pdf(): text-layer PDF (direct extraction, no OCR)

A PDF with an embedded text layer is extracted instantly by PyMuPDF.
Chrome Screen AI is **not** called because the text is already machine-readable.

In [ ]:
import fitz   # PyMuPDF — pip install pymupdf

doc = fitz.open()

p1 = doc.new_page(width=595, height=842)
p1.insert_text((60,  60), 'chrome-ocr Test Document', fontsize=24, fontname='helv')
p1.insert_text((60, 110), 'Page 1 — Introduction', fontsize=16)
p1.insert_text((60, 145),
    'This PDF has a text layer so PyMuPDF extracts text directly.\n'
    'The quick brown fox jumps over the lazy dog.\n'
    'No OCR is needed for this page.',
    fontsize=12)

p2 = doc.new_page(width=595, height=842)
p2.insert_text((60, 60), 'Page 2 — API Reference', fontsize=16)
p2.insert_text((60, 100),
    'ocr_pdf(path, dpi=200, pages=None)\n'
    'ocr_img(image)\n'
    'ScreenAIEngine(dll_path=None)',
    fontsize=12, fontname='cour')

doc.save('test_text.pdf')
doc.close()

md = ocr_pdf('test_text.pdf', engine=ENGINE)
print(md)

---
## 4 — ocr_pdf(): image-only PDF (scanned — OCR required)

Text is **rasterised into a PNG** and embedded in the PDF as an image.
The page has **no text layer**, so `page.get_text()` returns `''` and
`ocr_pdf()` falls through to Chrome Screen AI OCR.

We first confirm the page is truly text-free, then run OCR.

In [ ]:
# Build a high-resolution image (>=150 DPI equivalent for good OCR)
scan_img = Image.new('RGB', (1240, 440), 'white')
d2 = ImageDraw.Draw(scan_img)

d2.text((40,  20), 'Scanned Document — Page 1',              fill='black', font=load_font(44))
d2.text((40,  90), 'Invoice #2024-001',                       fill='black', font=load_font(30))
d2.text((40, 145), 'Item             Qty   Unit Price   Total', fill='black', font=load_font(24))
d2.text((40, 185), 'Widget A          10      $12.50   $125.00', fill='#222', font=load_font(22))
d2.text((40, 215), 'Widget B           5      $30.00   $150.00', fill='#222', font=load_font(22))
d2.text((40, 245), 'Service fee        1     $200.00   $200.00', fill='#222', font=load_font(22))
d2.text((40, 295), 'Total                               $475.00', fill='black', font=load_font(26))
d2.text((40, 365), 'Thank you for your business!',            fill='#444', font=load_font(20))

scan_img.save('test_scanned_page.png')
print('Scanned page image saved')

In [ ]:
# Embed as image-only PDF (no text layer)
buf = io.BytesIO()
scan_img.save(buf, format='PNG')
buf.seek(0)

doc2 = fitz.open()
page = doc2.new_page(width=595, height=842)
page.insert_image(fitz.Rect(20, 20, 575, 290), stream=buf.read())
doc2.save('test_scanned.pdf')
doc2.close()

# Verify no text layer exists
check = fitz.open('test_scanned.pdf')
direct_text = check[0].get_text().strip()
check.close()
print(f'Direct extraction returned: {direct_text!r}')   # expected: ''
print('=> ocr_pdf() will use Chrome Screen AI OCR\n')

md_scanned = ocr_pdf('test_scanned.pdf', engine=ENGINE)
print(md_scanned)

---
## 5 — Page selection and DPI control

In [ ]:
doc3 = fitz.open()
for i in range(1, 5):
    p = doc3.new_page(width=595, height=842)
    p.insert_text((60, 60),  f'Chapter {i}',                  fontsize=20, fontname='helv')
    p.insert_text((60, 100), f'Body text for chapter {i}.',    fontsize=12)
doc3.save('test_multipage.pdf')
doc3.close()

print('=== Pages 1 and 3 only ===')
print(ocr_pdf('test_multipage.pdf', pages=[1, 3], engine=ENGINE))

print('\n=== Pages 2-4 via range ===')
print(ocr_pdf('test_multipage.pdf', pages=range(2, 5), engine=ENGINE))

---
## 6 — Save to Markdown file

In [ ]:
md_full = ocr_pdf('test_text.pdf', engine=ENGINE)
with open('output.md', 'w', encoding='utf-8') as f:
    f.write(md_full)
print(f'Saved {len(md_full):,} chars to output.md')

---
## 7 — Summary: all input types with the shared engine

In [ ]:
import numpy as np

tasks = [
    ('image file path', lambda: ocr_img('test_image.png',   engine=ENGINE)),
    ('PIL Image',       lambda: ocr_img(img,                 engine=ENGINE)),
    ('NumPy array',     lambda: ocr_img(np.array(img),       engine=ENGINE)),
    ('text-layer PDF',  lambda: ocr_pdf('test_text.pdf',    pages=1, engine=ENGINE)),
    ('scanned PDF',     lambda: ocr_pdf('test_scanned.pdf', pages=1, engine=ENGINE)),
]

for label, fn in tasks:
    result  = fn()
    preview = result[:100].replace('\n', ' ')
    print(f'[{label:>16}]  {preview}...')
    print()